In [1]:
pip install langchain langchain_community langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 163.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 599.5/599.5 kB 30.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install shap

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import shap

In [2]:
classifier_features=['flgs_e', 'flgs_e s', 'flgs_e dS', 'flgs_e g', 'flgs_e *', 'flgs_eU',
       'flgs_e &', 'flgs_e    F', 'flgs_e r', 'proto_tcp', 'proto_udp',
       'proto_icmp', 'proto_arp', 'proto_ipv6-icmp', 'proto_rarp',
       'proto_igmp', 'saddr_192.168.100.149', 'saddr_192.168.100.148',
       'saddr_192.168.100.150', 'saddr_192.168.100.147', 'saddr_192.168.100.3',
       'saddr_192.168.100.7', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1', 'daddr_192.168.100.149', 'daddr_192.168.100.150',
       'daddr_192.168.100.147', 'daddr_192.168.100.3', 'daddr_192.168.100.7',
       'daddr_192.168.100.6', 'daddr_192.168.100.5', 'daddr_192.168.100.4',
       'daddr_private', 'daddr_external', 'pkts', 'bytes', 'state_RST',
       'state_CON', 'state_NRS', 'state_ACC', 'state_MAS', 'dur', 'mean',
       'min', 'dpkts', 'dbytes', 'tnp_per_dport', 'ar_p_proto_p_dstip',
       'ar_p_proto_p_sport', 'pkts_p_state_p_protocol_p_destip','attack_type']
classifier_data=pd.read_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/Classifier_Data/bot_iot_dataset_preprocessed.csv")[classifier_features]
classifier_data.head()

,flgs_e,flgs_e s,flgs_e dS,flgs_e g,flgs_e *,flgs_eU,flgs_e &,flgs_e F,flgs_e r,proto_tcp,...,dur,mean,min,dpkts,dbytes,tnp_per_dport,ar_p_proto_p_dstip,ar_p_proto_p_sport,pkts_p_state_p_protocol_p_destip,attack_type
0,1,0,0,0,0,0,0,0,0,1,...,0.004128,0.004128,0.004128,1,60,8,939.965691,564.567137,6482,3
1,0,1,0,0,0,0,0,0,0,1,...,31.771238,0.000000,0.000000,0,0,13108094,1254.250649,0.094425,12994,1
2,1,0,0,0,0,0,0,0,0,1,...,0.000224,0.000224,0.000224,1,60,21,1254.250649,4368.510051,13229,3
3,0,1,0,0,0,0,0,0,0,1,...,31.326099,0.017032,0.000000,2,120,13108094,9901.705447,0.191534,14111,1
4,1,0,0,0,0,0,0,0,0,1,...,0.000566,0.000566,0.000566,1,60,23,1254.250649,1856.201331,13229,3


In [3]:
classifier_data.iloc[0]

flgs_e                                 1.000000
flgs_e s                               0.000000
flgs_e dS                              0.000000
flgs_e g                               0.000000
flgs_e *                               0.000000
flgs_eU                                0.000000
flgs_e &                               0.000000
flgs_e    F                            0.000000
flgs_e r                               0.000000
proto_tcp                              1.000000
proto_udp                              0.000000
proto_icmp                             0.000000
proto_arp                              0.000000
proto_ipv6-icmp                        0.000000
proto_rarp                             0.000000
proto_igmp                             0.000000
saddr_192.168.100.149                  1.000000
saddr_192.168.100.148                  0.000000
saddr_192.168.100.150                  0.000000
saddr_192.168.100.147                  0.000000
saddr_192.168.100.3                    0

In [3]:
def get_key(dict,value) :
    return [key for key in dict.keys() if dict[key]==value][0]
def sort_get_best(shap_values,features) :
    feat_dict={features[i]:shap_values[i] for i in range(0,len(features))}
    sorted_shap = np.argsort(np.abs(shap_values))
    result_shap = shap_values[sorted_shap][::-1][:10]
    feature_importances={get_key(feat_dict,value):value for value in result_shap}
    return feature_importances

In [1]:
!pip install gdown

In [4]:
def make_prompt(data_row,classifier,metadata,best_shap_values) :
    classes={0:'Normal Traffic',1 : 'DoS attack',2: 'DDoS attack',3: 'OS Fingerprint attack',4: 'Service Scan attack',5 : 'Keylogging attack',6 : 'Data Exfiltration attack'}
    #making the prompts
    flags_state=['flgs_e', 'flgs_e s', 'flgs_e dS', 'flgs_e g', 'flgs_e *', 'flgs_eU',
        'flgs_e &', 'flgs_e    F', 'flgs_e r']
    protos=['proto_tcp', 'proto_udp',
        'proto_icmp', 'proto_arp', 'proto_ipv6-icmp', 'proto_rarp',
        'proto_igmp']
    s_addr=['saddr_192.168.100.149', 'saddr_192.168.100.148',
        'saddr_192.168.100.150', 'saddr_192.168.100.147', 'saddr_192.168.100.3',
        'saddr_192.168.100.7', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
        'saddr_192.168.100.1']
    d_addr=['daddr_192.168.100.149', 'daddr_192.168.100.150',
        'daddr_192.168.100.147', 'daddr_192.168.100.3', 'daddr_192.168.100.7',
        'daddr_192.168.100.6', 'daddr_192.168.100.5', 'daddr_192.168.100.4']
    d_type=['daddr_private','daddr_external']
    states=['state_RST','state_CON', 'state_NRS', 'state_ACC', 'state_MAS']
    transaction_info=['pkts', 'bytes','dur', 'mean',
        'min', 'dpkts', 'dbytes', 'tnp_per_dport', 'ar_p_proto_p_dstip',
        'ar_p_proto_p_sport', 'pkts_p_state_p_protocol_p_destip']

    #network traffic and classifier decision
    prompt="Generate a concise yet profound explanation of a network classifier's decision regarding whether the analyzed network traffic is normal or represents a type of attack. Use the provided SHAP values to highlight the classifier's reasoning while also considering key traffic details such as transaction state, source and destination addresses, and transaction-specific information. Ensure the explanation is insightful and well-rounded.\ninput : {\n"
    prompt=prompt+"Network Traffic :\n\n"
    columns=list(data_row.index)
    columns.append('additional_info')
    columns.append('attack_type')
    y_pred=classifier.predict([data_row])[0]
    for feature in columns :
        if feature=="additional_info" :
            prompt=prompt+f"\nConsider the following information as well: \n-The IP addresses 192.168.100.147, 192.168.100.148, 192.168.100.149, and 192.168.100.150 are identified as botnets, meaning that attacks are likely to originate from these IPs. \n-The IP address 192.168.100.3 corresponds to the server within the network which is usually targeted by network attacks.\n"
        if feature=="attack_type" :
            prompt=prompt+f"\nClassifier Decision : {classes[y_pred]}\n"
        elif feature in flags_state :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Flow state flags: {metadata[feature]}\n"
        elif feature in protos :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Transaction protocol : {metadata[feature]}\n"
        elif feature in s_addr :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Source IP address : {metadata[feature]}\n"
        elif feature in d_addr :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Destination IP address : {metadata[feature]}\n"
        elif feature in d_type :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Destination address type : {metadata[feature]}\n"
        elif feature in states :
            if data_row[feature]==1.0 :
                prompt=prompt+f"-Transaction State : {metadata[feature]}\n"
        elif feature in transaction_info :
            if "\nTransaction information :\n" in prompt :
                prompt=prompt+f"-{metadata[feature]} : {round(data_row[feature],2)}\n"
            else :
                prompt=prompt+"\nTransaction information :\n"
                prompt=prompt+f"-{metadata[feature]} : {round(data_row[feature],2)}\n"
    #shap values
    prompt=prompt+"\nSHAP Values that represent how much a particular feature influenced the final decision:\n"
    for feature in best_shap_values.keys() :
        if feature in flags_state :
            if data_row[feature]==1.0 :
                prompt=prompt+ f"-SHAP Value for Flow state flags : {metadata[feature]} : {round(best_shap_values[feature],2)} (Feature value: {data_row[feature]})\n"
        elif feature in protos :
            if data_row[feature]==1.0 :
                prompt=prompt+ f"-SHAP Value for Transaction protocol : {metadata[feature]} : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        elif feature in s_addr :
            if data_row[feature]==1.0 :
                prompt=prompt+ f"-SHAP Value for Source IP Address ({metadata[feature]}) : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        elif feature in d_addr :
            if data_row[feature]==1.0 :
                prompt=prompt+ f"-SHAP Value for Destination IP Address ({metadata[feature]}) : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        elif feature in d_type :
            prompt=prompt+ f"-SHAP Value for Destination Address Type ({metadata[feature]}) : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        elif feature in states :
            if data_row[feature]==1.0 :
                prompt=prompt+ f"-SHAP Value for Transaction state ({metadata[feature]}) : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        else :
            prompt=prompt+ f"-SHAP Value for {metadata[feature]} : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
    prompt=prompt[:-1]+'}'
    return prompt

In [5]:
def data_pipeline(data_row) :
    with open("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/prompt_features.json",'r') as f :
        metadata=json.load(f)
    classifier=joblib.load("/teamspace/studios/this_studio/classifier.pkl")
    explainer=shap.TreeExplainer(classifier)
    shap_values=explainer.shap_values([data_row])[0,:,classifier.predict([data_row])[0]]
    best_shap_values=sort_get_best(shap_values,data_row.index)
    return make_prompt(data_row,classifier,metadata,best_shap_values)

In [13]:
data_row=classifier_data.drop("attack_type",axis=1).iloc[0]
print(data_pipeline(data_row))

Generate a concise yet profound explanation of a network classifier's decision regarding whether the analyzed network traffic is normal or represents a type of attack. Use the provided SHAP values to highlight the classifier's reasoning while also considering key traffic details such as transaction state, source and destination addresses, and transaction-specific information. Ensure the explanation is insightful and well-rounded.
input : {
Network Traffic :

Flow state flags: State ESTABLISHED
Transaction protocol : TCP
Source IP address : 192.168.100.149
Destination IP address : 192.168.100.5
Destination address type : Private

Transaction information :
-Total count of packets in transaction : 2.0
-Total number of bytes in transaction : 120.0
-Transaction State : RESET
-Record total duration : 0.0
-Average duration of aggregated records : 0.0
-Minimum duration of aggregated records : 0.0
-Destination-to-source packet count : 1.0
-Destination-to-source byte count : 60.0
-Total Number o

In [32]:
def clean_response(response) :
    """
    Function for cleaning the repsonses generated by the ExplainerLLM
    """
    parts=response.split('}')
    output=''
    if len(parts)==2 :
        return parts[1].strip()
    elif len(parts)==3 :
       return parts[2].strip()
    elif len(parts)>3 :
       for i in range(2,len(parts)) :
           output+=parts[i]
           output+='}'
       return output[:-1].strip()

In [36]:
def clean_response_2(response):
    response=clean_response(response)
    if not response.endswith(('.', '!', '?')):
        last_period = max(response.rfind('.'), response.rfind('!'), response.rfind('?'))
        if last_period != -1:
            response = response[:last_period + 1]
    return response

In [37]:
print(clean_response_2(string))

Based on the provided SHAP values, the classifier decision of DDoS attack is likely justified. The SHAP values for Total number of bytes in transaction and Total number of packets per destination port are both significantly positive, indicating that these features have a substantial impact on the classifier's decision. The high volume of data transmitted (308 bytes) and the large number of packets (131,080,094) suggest that this network traffic is likely to be part of a DDoS attack.

The SHAP values for Record total duration and Total count of packets in transaction also contribute to the decision, although to a lesser extent. The short duration of the transaction (9.47 seconds) and the relatively small number of packets (2) suggest that this network traffic may not be a typical DDoS attack.

The SHAP values for Average rate per protocol per source port and Destination-to-source packet count are both negative, indicating that these features have a minimal impact on the classifier's dec

In [30]:
string="""Generate a concise yet profound explanation of a network classifier's decision regarding whether the analyzed network traffic is normal or represents a type of attack. Use the provided SHAP values to highlight the classifier's reasoning while also considering key traffic details such as transaction state, source and destination addresses, and transaction-specific information. Ensure the explanation is insightful and well-rounded.
input : {
Network Traffic :

Flow state flags: State ESTABLISHED and State SYN_SENT
Transaction protocol : TCP
Source IP address : 192.168.100.147
Destination IP address : 192.168.100.3
Destination address type : Private

Transaction information :
-Total count of packets in transaction : 2.0
-Total number of bytes in transaction : 308.0
-Record total duration : 9.47
-Average duration of aggregated records : 0.0
-Minimum duration of aggregated records : 0.0
-Destination-to-source packet count : 0.0
-Destination-to-source byte count : 0.0
-Total Number of packets per destination port : 13108094.0
-Average rate per protocol per Destination IP : 5274.17
-Average rate per protocol per source port : 0.11
-Number of packets grouped by state of flows and protocols per destination IP : 7648.0

Consider the following information as well: 
-The IP addresses 192.168.100.147, 192.168.100.148, 192.168.100.149, and 192.168.100.150 are identified as botnets, meaning that attacks are likely to originate from these IPs. 
-The IP address 192.168.100.3 corresponds to the server within the network which is usually targeted by network attacks.

Classifier Decision : DDoS attack

SHAP Values that represent how much a particular feature influenced the final decision:
-SHAP Value for Total number of bytes in transaction : 2.440000057220459. (Feature value: 308.0)
-SHAP Value for Total Number of packets per destination port : 2.0199999809265137. (Feature value: 13108094.0)
-SHAP Value for Record total duration : 1.5499999523162842. (Feature value: 9.473212)
-SHAP Value for Total count of packets in transaction : 1.1299999952316284. (Feature value: 2.0)
-SHAP Value for Average rate per protocol per source port : 0.4099999964237213. (Feature value: 0.105561)
-SHAP Value for Destination-to-source byte count : -0.3799999952316284. (Feature value: 0.0)
-SHAP Value for Destination-to-source packet count : -0.3100000023841858. (Feature value: 0.0)
-SHAP Value for Number of packets grouped by state of flows and protocols per destination IP : 0.27000001072883606. (Feature value: 7648.0)
-SHAP Value for Source IP Address (192.168.100.147) : -0.18000000715255737. (Feature value: 1.0)} 

Based on the provided SHAP values, the classifier decision of DDoS attack is likely justified. The SHAP values for Total number of bytes in transaction and Total number of packets per destination port are both significantly positive, indicating that these features have a substantial impact on the classifier's decision. The high volume of data transmitted (308 bytes) and the large number of packets (131,080,094) suggest that this network traffic is likely to be part of a DDoS attack.

The SHAP values for Record total duration and Total count of packets in transaction also contribute to the decision, although to a lesser extent. The short duration of the transaction (9.47 seconds) and the relatively small number of packets (2) suggest that this network traffic may not be a typical DDoS attack.

The SHAP values for Average rate per protocol per source port and Destination-to-source packet count are both negative, indicating that these features have a minimal impact on the classifier's decision. The low traffic volume from the source IP (192.168.100.147) and the absence of any traffic from the destination IP (192.168.100.3) also contribute to the decision, suggesting that this network traffic is unlikely to be a DDois attack.

The SHAP value for Source IP Address (192.168.100.147) is negative, indicating that this feature has a minimal impact on the classifier's decision. The fact that this IP address is identified as a botnet suggests that attacks are likely to originate from this IP, but the SHAP value indicates that the classifier is not strongly influenced by this feature.

The SHAP value for Number of packets grouped by state of flows and protocols per destination IP is also negative, indicating that this feature has a minimal impact on the classifier's decision. The fact that this feature is related to the destination IP (192.168.100.3) suggests that the classifier is not strongly influenced by this feature.

In conclusion, the classifier decision of DDoS attack is likely justified based on the provided SHAP values. The high volume of data transmitted and the large number of packets suggest that this network traffic is likely to be part of a DDoS attack. The short duration of the transaction and the relatively small number of packets suggest that this network traffic may not be a typical DDoS attack. The low traffic volume from the source IP and the absence of any traffic from the destination IP also contribute to the decision, suggesting that this network traffic is unlikely to be a DDoS attack.

However, it is worth noting that the SHAP values for Total number of bytes in transaction and Total number of packets per destination port are both significantly positive, indicating that these features have a substantial impact on the classifier's decision. This suggests that the classifier may be overestimating the impact of these features, and that other factors may be contributing to the decision.

Therefore, while the classifier decision of DDoS attack is likely justified, it is also important to consider other factors that may be contributing to the decision, such as the network configuration, the traffic patterns, and the security measures in place. A more nuanced understanding of the network traffic and the underlying factors that contribute to the decision is necessary to make a more accurate assessment. 

Note: The SHAP values are based on a hypothetical scenario, and the actual SHAP values may vary depending on the specific implementation and the data used. 

Also, the explanation is concise and provides a clear insight into the classifier's decision-making process. The use of SHAP values to highlight the classifier's reasoning and the consideration of key traffic details provide a well-rounded explanation. 

The explanation is also insightful, as it highlights the potential limitations of the classifier's decision and encourages further investigation into the underlying factors that contribute to the decision. 

Finally, the explanation is well-rounded, as it provides a clear and concise overview of the network traffic and the classifier's decision, while also considering the potential limitations and nuances of the decision. 

Overall,
"""

In [15]:
import sys
import time

for char in string:
    sys.stdout.write(char)
    sys.stdout.flush()
    time.sleep(0.005)

Based on the provided SHAP values, the network classifier has determined that the analyzed network traffic represents a Data Exfiltration attack. The classifier's reasoning is as follows:

* The transaction protocol is TCP, which is a common protocol used in legitimate network traffic, but it's not a deciding factor in this case.
* The source IP address (192.168.100.150) has a SHAP value of 1.1, indicating that it played a relatively minor role in the classifier's decision. However, considering the context of the network, it's possible that this IP address is involved in the attack, given that it's part of a botnet (192.168.100.147, 192.168.100.148, 192.168.100.149).
* The destination IP address (192.168.100.3) is a private IP address, which is often used for internal network traffic. However, in this case, it's being targeted by an attack, which suggests that something unusual is going on.
* The transaction information suggests that the transaction is relatively short-lived, with an a